# Part C — Evals harness
## Notebook 04

A migration is only trustworthy if it's **measured**. This notebook builds a labeled good/bad email set and
computes three metrics, persisting every run to `EVAL_RUNS`:

1. **Scoring accuracy** — does the model's intent score agree with the labels?
2. **Run-to-run consistency** — how stable is the score across repeated runs?
3. **Coached win-rate lift** — predicted win-rate gain from rewriting weak emails to the framework.

Ground truth uses the unambiguous quality tiers (tier 3 = good/high-intent, tier 1 = poor/low-intent).

**Prerequisite:** `gtm-03-after-agents` Checkpoint B PASSED.

---
## Section 1: Connect

In [ ]:
from snowflake.snowpark.context import get_active_session
import uuid
session = get_active_session()
for s in ['USE ROLE SYSADMIN','USE DATABASE GTMAGENTS','USE SCHEMA GTMAGENTS.DEMO','USE WAREHOUSE GTMAGENTS_WH']:
    session.sql(s).collect()
print('Connected')

---
## Section 2: Build the labeled set

`EMAIL_LABELS` = a balanced sample of clearly-good (tier 3 → label 1) and clearly-poor (tier 1 → label 0)
emails. This is the gold set every eval scores against.

In [ ]:
session.sql("""
    CREATE OR REPLACE TABLE EMAIL_LABELS AS
    SELECT id, body, IFF(quality_tier = 3, 1, 0) AS label
    FROM (
        SELECT id, body, quality_tier,
               ROW_NUMBER() OVER (PARTITION BY quality_tier ORDER BY RANDOM()) AS rn
        FROM EMAILS WHERE quality_tier IN (1, 3)
    ) WHERE rn <= 30
""").collect()
session.sql("SELECT label, COUNT(*) AS n FROM EMAIL_LABELS GROUP BY label ORDER BY label").show()

---
## Section 3: Scoring accuracy + run-to-run consistency

We score the labeled set twice with the cheap model. **Accuracy** = agreement between (score >= 0.5) and the
label. **Consistency** = share of emails whose two runs land on the same side of the 0.5 threshold. Both runs
are persisted to `EVAL_RUNS`.

In [ ]:
def score_labeled(run_tag):
    session.sql(f"""
        CREATE OR REPLACE TEMPORARY TABLE _eval_{run_tag} AS
        SELECT id, label,
            LEAST(1, GREATEST(0, COALESCE(TRY_CAST(REGEXP_SUBSTR(
                AI_COMPLETE('llama3.1-8b', 'Output ONLY a number 0.00-1.00 for the buyer intent of this email: ' || body),
                '[0-9]*\\.?[0-9]+') AS FLOAT), 0.5))) AS score
        FROM EMAIL_LABELS
    """).collect()

score_labeled('a')
score_labeled('b')

row = session.sql("""
    SELECT
      ROUND(AVG(IFF((a.score >= 0.5) = (a.label = 1), 1, 0)), 4) AS accuracy,
      ROUND(AVG(IFF((a.score >= 0.5) = (b.score >= 0.5), 1, 0)), 4) AS consistency,
      COUNT(*) AS n
    FROM _eval_a a JOIN _eval_b b ON a.id = b.id
""").collect()[0]
accuracy, consistency, n = row['ACCURACY'], row['CONSISTENCY'], row['N']
print(f'accuracy={accuracy}  consistency={consistency}  n={n}')

---
## Section 4: Coached win-rate lift

Rewriting a weak email to the framework should raise its intent — and, using the observed relationship between
quality and outcomes, its **predicted win rate**. We take poor (tier-1) emails, coach-rewrite a sample with the
model, re-score, and translate the intent gain into a win-rate lift using the observed tier win rates.

> Kept to a small sample so it runs interactively.

In [ ]:
# Observed win rate for poor vs good emails — the anchor for the lift translation.
wr = {r['QUALITY_TIER']: r['WR'] for r in session.sql("""
    SELECT e.quality_tier, AVG(IFF(o.won,1,0)) AS wr
    FROM EMAILS e JOIN OUTCOMES o ON o.email_id=e.id WHERE e.quality_tier IN (1,3) GROUP BY e.quality_tier
""").collect()}

# Coach-rewrite a sample of poor emails and re-score original vs rewrite.
lift = session.sql(f"""
    WITH sample AS (SELECT id, body FROM EMAILS WHERE quality_tier=1 ORDER BY RANDOM() LIMIT 15),
    scored AS (
      SELECT id,
        LEAST(1,GREATEST(0,COALESCE(TRY_CAST(REGEXP_SUBSTR(AI_COMPLETE('llama3.1-8b','Output ONLY a number 0.00-1.00 for the buyer intent of this email: '||body),'[0-9]*\\.?[0-9]+') AS FLOAT),0.5))) AS orig_score,
        LEAST(1,GREATEST(0,COALESCE(TRY_CAST(REGEXP_SUBSTR(AI_COMPLETE('llama3.1-8b','Output ONLY a number 0.00-1.00 for the buyer intent of this REWRITTEN email: '||
          AI_COMPLETE('mistral-large2','Rewrite this cold email to be personalized, with a clear single CTA, under 120 words, with a quantified value prop and an in-market signal. Return only the rewritten email: '||body)),'[0-9]*\\.?[0-9]+') AS FLOAT),0.5))) AS coached_score
      FROM sample)
    SELECT ROUND(AVG(coached_score - orig_score),4) AS intent_lift FROM scored
""").collect()[0]['INTENT_LIFT']

# Translate intent lift to predicted win-rate lift (scaled by the good-vs-poor win-rate gap).
winrate_gap = (wr.get(3,0.06) - wr.get(1,0.001))
winrate_lift = round((lift or 0) * winrate_gap, 5)
print(f'avg intent lift={lift}  ->  predicted win-rate lift={winrate_lift}')

---
## Section 5: Persist runs to `EVAL_RUNS`

In [ ]:
run_id = str(uuid.uuid4())[:8]
session.sql(f"""INSERT INTO EVAL_RUNS (run_id, model_name, accuracy, consistency, winrate_lift, n_examples)
    VALUES ('{run_id}-a','llama3.1-8b',{accuracy},{consistency},{winrate_lift},{n})""").collect()
session.sql(f"""INSERT INTO EVAL_RUNS (run_id, model_name, accuracy, consistency, winrate_lift, n_examples)
    VALUES ('{run_id}-b','llama3.1-8b',{accuracy},{consistency},{winrate_lift},{n})""").collect()
session.sql('SELECT run_id, model_name, accuracy, consistency, winrate_lift, n_examples FROM EVAL_RUNS ORDER BY run_ts DESC LIMIT 10').show()

---
## Section 6 (optional): managed loop with Cortex AI Function Studio

For a fully managed create → evaluate → optimize loop (and a cost/quality Pareto comparison that finds the
cheapest model matching accuracy), register the scorer as a Cortex AI Function and use the Studio SPROCs.
These are **long-running** — run them live in the demo, not inline here:

```sql
-- Register (9 positional params): CALL SNOWFLAKE.CORTEX.CREATE_AI_FUNCTION(...);
-- Evaluate (12 positional params): CALL SNOWFLAKE.CORTEX.EVALUATE_AI_FUNCTION(...);
-- Optimize (18 positional params): CALL SNOWFLAKE.CORTEX.OPTIMIZE_AI_FUNCTION(...);
```

See the `cortex-ai-function-studio` skill for exact signatures. The harness above already gives you accuracy,
consistency, and win-rate lift without the managed job.

---
## ✅ Checkpoint C

All must PASS before Part D.

In [ ]:
checks = []
n_runs = session.sql('SELECT COUNT(*) AS n FROM EVAL_RUNS').collect()[0]['N']
checks.append(('EVAL_RUNS has >= 2 runs', n_runs >= 2))
nulls = session.sql('SELECT COUNT(*) AS n FROM EVAL_RUNS WHERE accuracy IS NULL OR consistency IS NULL').collect()[0]['N']
checks.append(('accuracy/consistency non-null', nulls == 0))
lab = session.sql('SELECT COUNT(DISTINCT label) AS n FROM EMAIL_LABELS').collect()[0]['N']
checks.append(('labeled set has both classes', lab == 2))

print('CHECKPOINT C')
for name, ok in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
overall = all(ok for _, ok in checks)
print()
print('RESULT:', 'PASS — proceed to Part D' if overall else 'FAIL — fix above before Part D')
assert overall, 'Checkpoint C failed'

---
## Summary

The evals harness quantifies the migration: scoring accuracy vs labels, run-to-run consistency, and predicted
win-rate lift from coached drafts — all persisted to `EVAL_RUNS`. Next, **`app/streamlit_app.py`** turns these
tables plus the in-plane traces into a live observability + comparison dashboard (Parts D & E).